In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split

In [14]:
class AgeGenderCNN(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super(AgeGenderCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)
        )

        # Flatten sonrası boyutu otomatik hesapla
        dummy_input = torch.zeros(1, 1, *input_size)
        dummy_output = self.features(dummy_input)
        self.flattened_size = dummy_output.view(1, -1).shape[1]

        # Cinsiyet tahmini başlığı
        self.gender_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

        # Yaş tahmini başlığı
        self.age_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        gender_out = torch.sigmoid(self.gender_head(x))
        age_out = self.age_head(x)
        return gender_out, age_out


In [15]:
class AgeGenderDataset(Dataset):
    def __init__(self, X, y_gender, y_age):
        self.X = X.astype(np.float32) / 255.0
        self.y_gender = y_gender.astype(np.float32).reshape(-1, 1)
        self.y_age = y_age.astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)
        return torch.tensor(x), torch.tensor(self.y_gender[idx]), torch.tensor(self.y_age[idx])

In [16]:
def train_model(model, dataloader, criterion_g, criterion_a, optimizer, device):
    model.train()
    total_loss = 0
    for x, y_g, y_a in tqdm(dataloader):
        x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
        optimizer.zero_grad()
        pred_g, pred_a = model(x)
        loss_g = criterion_g(pred_g, y_g)
        loss_a = criterion_a(pred_a, y_a)
        loss = loss_g + loss_a
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [17]:
def evaluate_model(model, dataloader, criterion_g, criterion_a, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
            pred_g, pred_a = model(x)
            loss_g = criterion_g(pred_g, y_g)
            loss_a = criterion_a(pred_a, y_a)
            loss = loss_g + loss_a
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [18]:
def test_model(model, dataloader, device):
    model.eval()
    preds_g, preds_a, true_g, true_a = [], [], [], []
    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x = x.to(device)
            pred_g, pred_a = model(x)
            preds_g += pred_g.cpu().numpy().flatten().tolist()
            preds_a += pred_a.cpu().numpy().flatten().tolist()
            true_g += y_g.numpy().flatten().tolist()
            true_a += y_a.numpy().flatten().tolist()

    preds_g_bin = [1 if p > 0.5 else 0 for p in preds_g]
    acc = accuracy_score(true_g, preds_g_bin)
    mae = mean_absolute_error(true_a, preds_a)
    return acc, mae

In [19]:
X_train = np.load("X_train_utkface.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_utkface.npy")
y_gen_train = np.load("y_gender_train_utkface.npy")

X_test = np.load("X_test_utkface.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_utkface.npy")
y_gen_test = np.load("y_gender_test_utkface.npy")
valid_mask = (y_gen_train == 0) | (y_gen_train == 1)

X_train = X_train[valid_mask]
y_gen_train = y_gen_train[valid_mask]
y_age_train = y_age_train[valid_mask]

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, random_state=42)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeGenderCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_utk3.pth"
best_val_loss = float('inf')
for epoch in range(100):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")
    

100%|██████████| 536/536 [00:10<00:00, 50.64it/s]


Epoch 0: Train Loss = 365.4347 | Val Loss = 373.0726
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:15<00:00, 34.34it/s]


Epoch 1: Train Loss = 291.8188 | Val Loss = 256.1710
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:23<00:00, 23.27it/s]


...

Epoch 97: Train Loss = 22.4236 | Val Loss = 164.8112


100%|██████████| 536/536 [00:18<00:00, 29.58it/s]


Epoch 98: Train Loss = 21.6644 | Val Loss = 164.6038


100%|██████████| 536/536 [00:17<00:00, 31.24it/s]


Epoch 99: Train Loss = 21.4318 | Val Loss = 193.2787


In [20]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN().to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
accuracy, mae = test_model(model, test_dataloader, device)
print(f"Test Seti Sonuçları:")
print(f"Cinsiyet Tahmin Doğruluğu: {accuracy * 100:.2f}%")
print(f"Yaş Tahmini Ortalama Mutlak Hata (MAE): {mae:.2f} yıl")

Test Seti Sonuçları:
Cinsiyet Tahmin Doğruluğu: 78.12%
Yaş Tahmini Ortalama Mutlak Hata (MAE): 9.08 yıl


In [21]:
X_train = np.load("X_train_all_imdbwiki.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_all_imdbwiki.npy")
y_gen_train = np.load("y_gender_train_all_imdbwiki.npy")

X_test = np.load("X_test_all_imdbwiki.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_all_imdbwiki.npy")
y_gen_test = np.load("y_gender_test_all_imdbwiki.npy")

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, random_state=42)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderCNN(input_size=(64, 64)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_imdbwiki3.pth"
best_val_loss = float('inf')
for epoch in range(100):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")

100%|██████████| 11459/11459 [02:06<00:00, 90.52it/s] 


Epoch 0: Train Loss = 186.9167 | Val Loss = 173.7696
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [01:54<00:00, 99.77it/s] 


Epoch 1: Train Loss = 164.8184 | Val Loss = 141.6006
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [01:53<00:00, 100.64it/s]


...

Epoch 97: Train Loss = 53.0259 | Val Loss = 166.4924


100%|██████████| 11459/11459 [01:17<00:00, 147.75it/s]


Epoch 98: Train Loss = 53.1050 | Val Loss = 182.3776


100%|██████████| 11459/11459 [01:20<00:00, 142.54it/s]


Epoch 99: Train Loss = 52.8899 | Val Loss = 165.4997


In [22]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN(input_size=(64, 64)).to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
accuracy, mae = test_model(model, test_dataloader, device)
print(f"Test Seti Sonuçları:")
print(f"Cinsiyet Tahmin Doğruluğu: {accuracy * 100:.2f}%")
print(f"Yaş Tahmini Ortalama Mutlak Hata (MAE): {mae:.2f} yıl")

Test Seti Sonuçları:
Cinsiyet Tahmin Doğruluğu: 75.11%
Yaş Tahmini Ortalama Mutlak Hata (MAE): 8.88 yıl
